# Metrik Retrieval (Terjangkau) & Eksperimen Ablasi
**Skripsi — Deteksi Persamaan pada Pokoknya Nama Merek Dagang**

Menjawab dua pertanyaan yang muncul dari hasil D–F:
1. **MAP/MRR tampak kecil** → karena 8 target tak-terjangkau ikut dihitung. Notebook ini menghitung
   metrik retrieval pada **query terjangkau saja** dan menampilkan berdampingan dgn versi semua-query.
2. **Bobot fonetik kecil (0.1)** → diuji lewat **ablasi**: bandingkan hybrid penuh vs tanpa-fonetik vs
   tanpa-semantik vs single-method, plus **fonID vs noFonID** (dengan/tanpa normalisasi fonetik Indonesia).

Input dari Langkah B–C: `haystack_preprocessed_fonID.csv`, `haystack_emb.npy`, `haystack_emb_index.csv`,
`gold_cases_categorized.csv`, dan model `cc.id.100.bin` di Drive.

> Catatan: fonID vs noFonID dihitung **on-the-fly** (Double Metaphone dengan/tanpa aturan digraf
> Indonesia), jadi TIDAK perlu file preprocessing terpisah.


## 0. Setup & muat

In [1]:
!pip install rapidfuzz metaphone fasttext-wheel scikit-learn -q
import os, re, numpy as np, pandas as pd
from rapidfuzz import process
from rapidfuzz.distance import JaroWinkler
from metaphone import doublemetaphone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
np.random.seed(42)
print("Setup selesai.")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 28.2 MB/s eta 0:00:00
Setup selesai.


In [2]:
butuh=["haystack_preprocessed_fonID.csv","haystack_emb.npy","haystack_emb_index.csv","gold_cases_categorized.csv"]
if not all(os.path.exists(f) for f in butuh):
    from google.colab import files
    print("Upload:", [f for f in butuh if not os.path.exists(f)]); files.upload()
hay=pd.read_csv("haystack_preprocessed_fonID.csv",dtype=str).fillna("")
hay=hay[hay["nama_base"].str.len()>0].reset_index(drop=True)
gold=pd.read_csv("gold_cases_categorized.csv",dtype=str).fillna("")
emb=np.load("haystack_emb.npy"); emb_index=pd.read_csv("haystack_emb_index.csv",dtype=str).fillna("")
SEM_DIM=emb.shape[1]
print("Haystack:",hay.shape,"| emb:",emb.shape)

Upload: ['gold_cases_categorized.csv']


Saving gold_cases_categorized.csv to gold_cases_categorized.csv
Haystack: (13514, 11) | emb: (13514, 100)


In [3]:
from google.colab import drive
drive.mount("/content/drive")
sem_model=None
try:
    import fasttext
    sem_model=fasttext.load_model("/content/drive/MyDrive/skripsi_merek/cc.id.100.bin")
    print("Model semantik dimuat.")
except Exception as e:
    print("Model semantik tak dimuat (semantik=0):",repr(e))
def sem_vec(t):
    if sem_model is None or not str(t).strip(): return None
    return sem_model.get_sentence_vector(str(t).replace("\n"," "))

Mounted at /content/drive
Model semantik dimuat.


## 1. Fungsi fitur (fonetik dgn/tanpa normalisasi ID)

In [4]:
def jw_sim(a,b): return JaroWinkler.normalized_similarity(a,b) if a and b else 0.0

def phon_norm(base, aktif=True):
    if not aktif: return base
    x=base
    for a,b in [("sy","sh"),("ny","n"),("kh","h")]: x=x.replace(a,b)
    return re.sub(r"([aeiou])\1+", r"\1", x)

def dm(t): return "".join(doublemetaphone(x)[0] for x in t.split())

def phon_sim(a_base,b_base,aktif=True):
    da,db=dm(phon_norm(a_base,aktif)),dm(phon_norm(b_base,aktif))
    return JaroWinkler.normalized_similarity(da,db) if da and db else 0.0

def sem_sim(a,b):
    va,vb=sem_vec(a),sem_vec(b)
    if va is None or vb is None: return 0.0
    na,nb=np.linalg.norm(va),np.linalg.norm(vb)
    return float(np.dot(va,vb)/(na*nb)) if na and nb else 0.0

## 2. Bangun pasangan berlabel + hitung semua fitur

In [5]:
ge=gold[gold["kategori_eval"]=="bermakna"].reset_index(drop=True)
rows=[]
for i,r in ge.iterrows():
    rows.append({"grup":f"c{i}","ab":r["merek_a_base"],"bb":r["merek_b_base"],
                 "y":1 if r["label_usulan"]=="mirip" else 0})
hu=hay[["nama_base"]].drop_duplicates().reset_index(drop=True)
for x in [r for r in rows if r["y"]==1]:
    g=0;t=0
    while g<3 and t<30:
        c=hu.sample(1).iloc[0]["nama_base"]; t+=1
        if jw_sim(x["ab"],c)<0.6: rows.append({"grup":x["grup"],"ab":x["ab"],"bb":c,"y":0}); g+=1
P=pd.DataFrame(rows)
P["jw"]=P.apply(lambda r:jw_sim(r["ab"],r["bb"]),axis=1)
P["phon_on"]=P.apply(lambda r:phon_sim(r["ab"],r["bb"],True),axis=1)
P["phon_off"]=P.apply(lambda r:phon_sim(r["ab"],r["bb"],False),axis=1)
P["sem"]=P.apply(lambda r:sem_sim(r["ab"],r["bb"]),axis=1)
print("Pasangan:",len(P),"| pos:",(P.y==1).sum(),"| neg:",(P.y==0).sum())

Pasangan: 196 | pos: 47 | neg: 149


## 3. Ablasi klasifikasi (7 konfigurasi, 5-fold CV)

In [6]:
def gw(step=0.1):
    r=[round(i*step,4) for i in range(int(round(1/step))+1)]; out=[]
    for a in r:
        for b in r:
            c=round(1-a-b,4)
            if -1e-9<=c<=1+1e-9: out.append((a,b,c))
    return out
THS=np.linspace(0.3,0.95,27); sgkf=StratifiedGroupKFold(5,shuffle=True,random_state=42)
y=P["y"].values; g=P["grup"].values

def cv_config(cols):
    X=P[cols].values; f1s,recs,aucs=[],[],[]
    for tr,te in sgkf.split(X,y,g):
        if X.shape[1]==1:
            th=max(THS,key=lambda t:f1_score(y[tr],(X[tr,0]>=t).astype(int),zero_division=0)); s=X[te,0]
        else:
            combos=gw() if X.shape[1]==3 else [(round(i*0.1,4),round(1-i*0.1,4)) for i in range(11)]
            best=None
            for w in combos:
                for th in THS:
                    s=sum(w[k]*X[tr,k] for k in range(len(w))); f1=f1_score(y[tr],(s>=th).astype(int),zero_division=0)
                    if best is None or f1>best[0]: best=(f1,w,th)
            _,w,th=best; s=sum(w[k]*X[te,k] for k in range(len(w)))
        pred=(s>=th).astype(int)
        f1s.append(f1_score(y[te],pred,zero_division=0)); recs.append(recall_score(y[te],pred,zero_division=0))
        try: aucs.append(roc_auc_score(y[te],s))
        except: pass
    return round(np.mean(f1s),3),round(np.mean(recs),3),round(np.mean(aucs) if aucs else float("nan"),3)

konfig={
 "Hybrid (fonID)":["jw","phon_on","sem"],
 "Hybrid (noFonID)":["jw","phon_off","sem"],
 "Tanpa fonetik (JW+sem)":["jw","sem"],
 "Tanpa semantik (JW+phon)":["jw","phon_on"],
 "JW saja":["jw"],
 "Fonetik saja":["phon_on"],
 "Semantik saja":["sem"],
}
tab=[]
for nama,cols in konfig.items():
    f1,rc,auc=cv_config(cols); tab.append({"Konfigurasi":nama,"F1":f1,"Recall":rc,"AUC":auc})
abl=pd.DataFrame(tab)
print(abl.to_string(index=False))
abl.to_csv("ablasi_klasifikasi.csv",index=False)

             Konfigurasi    F1  Recall   AUC
          Hybrid (fonID) 0.861   0.853 0.968
        Hybrid (noFonID) 0.861   0.853 0.968
  Tanpa fonetik (JW+sem) 0.859   0.856 0.972
Tanpa semantik (JW+phon) 0.812   0.741 0.925
                 JW saja 0.839   0.768 0.911
            Fonetik saja 0.758   0.743 0.914
           Semantik saja 0.685   0.678 0.874


## 4. Interpretasi ablasi otomatis (berdasarkan angka nyata)

In [7]:
base=abl.set_index("Konfigurasi")
def selisih(a,b,m="F1"): return round(base.loc[a,m]-base.loc[b,m],3)

print("=== Interpretasi berbasis data ===")
d_fon = selisih("Hybrid (fonID)","Tanpa fonetik (JW+sem)")
print(f"1. Kontribusi dimensi FONETIK  : hybrid - (tanpa fonetik) = {d_fon:+.3f} F1")
print("   ->", "fonetik MENAMBAH sinyal (berkontribusi meski bobot kecil)." if d_fon>0.005
      else "fonetik nyaris tak menambah -> REDUNDAN dengan tekstual (temuan sah).")
d_sem = selisih("Hybrid (fonID)","Tanpa semantik (JW+phon)")
print(f"2. Kontribusi dimensi SEMANTIK : hybrid - (tanpa semantik) = {d_sem:+.3f} F1")
print("   ->", "semantik MENAMBAH sinyal." if d_sem>0.005 else "semantik nyaris tak menambah.")
d_norm = selisih("Hybrid (fonID)","Hybrid (noFonID)")
print(f"3. Efek NORMALISASI FONETIK ID : fonID - noFonID = {d_norm:+.3f} F1")
print("   ->", "normalisasi fonetik Indonesia MEMBANTU." if d_norm>0.005
      else "normalisasi fonetik ID tak berdampak signifikan pada data ini.")
d_hyb = round(base.loc["Hybrid (fonID)","F1"] - base[["F1"]].loc[["JW saja","Fonetik saja","Semantik saja"]].max().iloc[0],3)
print(f"4. Hybrid vs single-method terbaik: selisih F1 = {d_hyb:+.3f}")
print("   ->", "HYBRID unggul atas metode tunggal." if d_hyb>0 else "hybrid belum mengungguli single-method terbaik (bahas jujur).")

=== Interpretasi berbasis data ===
1. Kontribusi dimensi FONETIK  : hybrid - (tanpa fonetik) = +0.002 F1
   -> fonetik nyaris tak menambah -> REDUNDAN dengan tekstual (temuan sah).
2. Kontribusi dimensi SEMANTIK : hybrid - (tanpa semantik) = +0.049 F1
   -> semantik MENAMBAH sinyal.
3. Efek NORMALISASI FONETIK ID : fonID - noFonID = +0.000 F1
   -> normalisasi fonetik ID tak berdampak signifikan pada data ini.
4. Hybrid vs single-method terbaik: selisih F1 = +0.022
   -> HYBRID unggul atas metode tunggal.


## 5. Metrik RETRIEVAL: versi semua-query vs terjangkau-saja
Memperbaiki kesan 'MAP kecil' — 8 target tak-terjangkau menyeret rata-rata. Di sini dilaporkan keduanya.
Bobot final diambil dari grid search di seluruh pasangan (konfigurasi hybrid fonID).


In [8]:
# bobot final hybrid fonID (refit seluruh data)
best=None
X=P[["jw","phon_on","sem"]].values
for w in gw():
    for th in THS:
        s=w[0]*X[:,0]+w[1]*X[:,1]+w[2]*X[:,2]; f1=f1_score(y,(s>=th).astype(int),zero_division=0)
        if best is None or f1>best[0]: best=(f1,w,th)
_,WF,THF=best
print(f"Bobot final: JW={WF[0]} Fonetik={WF[1]} Semantik={WF[2]} | threshold={THF:.2f}")

# precompute kandidat
cb=hay["nama_base"].tolist()
cd=hay["nama_phon"].apply(lambda t:"".join(doublemetaphone(x)[0] for x in t.split())).tolist()
id2row={rid:i for i,rid in enumerate(emb_index["id"].tolist())}
cemb=np.vstack([emb[id2row[i]] if i in id2row else np.zeros(SEM_DIM) for i in hay["id"]]).astype("float32")
cemb_n=cemb/(np.linalg.norm(cemb,axis=1,keepdims=True)+1e-9)

def retr(qb,qp):
    qdm="".join(doublemetaphone(x)[0] for x in qp.split())
    jw=process.cdist([qb],cb,scorer=JaroWinkler.normalized_similarity)[0]
    ph=process.cdist([qdm],cd,scorer=JaroWinkler.normalized_similarity)[0]
    qv=sem_vec(qb)
    se=(cemb_n@(qv/(np.linalg.norm(qv)+1e-9))) if qv is not None else np.zeros(len(cb))
    return WF[0]*jw+WF[1]*ph+WF[2]*se

mirip=ge[ge["label_usulan"]=="mirip"].reset_index(drop=True)
ranks_all=[]; ranks_reach=[]
for _,r in mirip.iterrows():
    sc=retr(r["merek_a_base"],r["merek_a_phon"]); order=np.argsort(-sc); rank=None
    for k,i in enumerate(order,1):
        if cb[i]==r["merek_b_base"] and cb[i]!=r["merek_a_base"]: rank=k; break
    if rank is None: ranks_all.append(None)
    else: ranks_all.append(rank); ranks_reach.append(rank)

def metr(ranks, n_total):
    ok=np.array([r for r in ranks if r is not None])
    out={}
    for k in [1,5,10,20]:
        out[f"Recall@{k}"]=round(sum(1 for r in ok if r<=k)/n_total,3)
    out["MRR"]=round(sum(1/r for r in ok)/n_total,3)
    out["MAP"]=out["MRR"]
    return out

n_all=len(mirip); n_reach=len(ranks_reach)
print(f"\nQuery mirip: {n_all} | terjangkau: {n_reach} | tak-terjangkau: {n_all-n_reach}")
m_all=metr([r for r in ranks_all],n_all)
m_reach=metr(ranks_reach,n_reach)
comp=pd.DataFrame({"Semua query (n=%d)"%n_all:m_all, "Terjangkau saja (n=%d)"%n_reach:m_reach})
print("\n=== Metrik Retrieval: semua vs terjangkau ===")
print(comp.to_string())
print(f"\nMedian rank target (terjangkau): {np.median(ranks_reach):.0f}")
comp.to_csv("retrieval_semua_vs_terjangkau.csv")

Bobot final: JW=0.5 Fonetik=0.1 Semantik=0.4 | threshold=0.55

Query mirip: 47 | terjangkau: 39 | tak-terjangkau: 8

=== Metrik Retrieval: semua vs terjangkau ===
           Semua query (n=47)  Terjangkau saja (n=39)
Recall@1                0.021                   0.026
Recall@5                0.468                   0.564
Recall@10               0.511                   0.615
Recall@20               0.553                   0.667
MRR                     0.234                   0.282
MAP                     0.234                   0.282

Median rank target (terjangkau): 3


## 6. Ringkasan untuk BAB IV
File tersimpan: `ablasi_klasifikasi.csv`, `retrieval_semua_vs_terjangkau.csv`.

**Poin yang bisa langsung ditulis (isi angka dari output di atas):**
- Metrik retrieval "terjangkau saja" adalah kinerja model murni; selisih dengan "semua query"
  = dampak keterbatasan cakupan haystack (8 target tak ditemukan) — dilaporkan jujur.
- Ablasi menunjukkan kontribusi tiap dimensi: lihat selisih F1 pada Bagian 4. Bobot fonetik kecil
  bila selisih 'tanpa fonetik' mendekati 0 → fonetik REDUNDAN dengan tekstual (bukan tak berguna).
- Efek normalisasi fonetik Indonesia = selisih fonID vs noFonID.
- Klaim keunggulan hybrid didukung bila F1 hybrid > single-method terbaik.
